In [ ]:
# Databricks notebook source
# Notebook: bcb_exchange_rates_ingest
# Purpose: Ingests the BCB USD/BRL exchange rate (series 10813) into raw_exchange_rates via MERGE,
#          and records each run in the run-history table read by check_bcb_consecutive_failures.

import time
import requests
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, lit, regexp_replace, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DecimalType
from grupo10.utils.logger import get_logger
from grupo10.utils.alerting import ensure_run_history_table, record_run_result

spark = SparkSession.builder.getOrCreate()

# 1. Catalog and schema parameters
dbutils.widgets.text("catalog", "lh_nautical")
dbutils.widgets.text("schema", "prod")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

target_table = f"{catalog}.{schema}.raw_exchange_rates"

# 2. Run tracking (consumed by the check_bcb_consecutive_failures task)
logger = get_logger("bcb_exchange_rates_ingest")
TASK_NAME = "ingest_bcb_exchange_rates"
RUN_HISTORY_TABLE = f"{catalog}.{schema}._alert_run_history"

ensure_run_history_table(spark, RUN_HISTORY_TABLE)

start = time.time()

try:
    # 3. Incremental fetch window
    if spark.catalog.tableExists(target_table):
        start_date = (datetime.now() - timedelta(days=15)).strftime("%d/%m/%Y")
    else:
        start_date = "01/01/2010"

    end_date = datetime.now().strftime("%d/%m/%Y")

    # 4. Call the BCB API (series 10813: USD commercial selling rate)
    # Splits the range into windows of up to 10 years (API limit) and sends headers
    # to avoid the 406 Not Acceptable error.
    HEADERS = {
        "User-Agent": "Mozilla/5.0 (compatible; DatabricksJob/1.0)",
        "Accept": "application/json"
    }

    def fetch_bcb_series(codigo, start_date_str, end_date_str, headers, max_years=10):
        """Fetches a BCB series, splitting the period into windows of at most `max_years` years."""
        start = datetime.strptime(start_date_str, "%d/%m/%Y")
        end = datetime.strptime(end_date_str, "%d/%m/%Y")
        all_data = []

        window_start = start
        while window_start <= end:
            window_end = min(window_start + timedelta(days=365 * max_years - 1), end)

            url = (
                f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
                f"?formato=json"
                f"&dataInicial={window_start.strftime('%d/%m/%Y')}"
                f"&dataFinal={window_end.strftime('%d/%m/%Y')}"
            )

            resp = requests.get(url, headers=headers, timeout=30)
            resp.raise_for_status()
            chunk = resp.json()
            all_data.extend(chunk)

            window_start = window_end + timedelta(days=1)

        return all_data

    data = fetch_bcb_series(10813, start_date, end_date, HEADERS)

    if not data:
        dbutils.notebook.exit(f"No data returned by the BCB API for the period {start_date} to {end_date}.")

    # 5. Process and structure the data with PySpark
    schema_json = StructType([
        StructField("data", StringType(), True),
        StructField("valor", StringType(), True)
    ])

    df_raw = spark.createDataFrame(data, schema=schema_json)

    df_transformed = df_raw.select(
        to_date(col("data"), "dd/MM/yyyy").alias("exchange_date"),
        lit("USD").alias("currency_from"),
        lit("BRL").alias("currency_to"),
        regexp_replace(col("valor"), ",", ".").cast(DecimalType(10, 4)).alias("exchange_rate"),
        current_timestamp().alias("_ingested_at")
    ).filter(col("exchange_date").isNotNull())

    # 6. Create or update (MERGE) in Delta Lake
    df_transformed.createOrReplaceTempView("stg_bcb_updates")

    create_sql = f"""
        CREATE TABLE IF NOT EXISTS {target_table} (
            exchange_date DATE,
            currency_from STRING,
            currency_to STRING,
            exchange_rate DECIMAL(10,4),
            _ingested_at TIMESTAMP
        ) USING DELTA
    """
    spark.sql(create_sql)

    merge_sql = f"""
        MERGE INTO {target_table} target
        USING stg_bcb_updates source
        ON target.exchange_date = source.exchange_date
       AND target.currency_from = source.currency_from
        WHEN MATCHED THEN UPDATE SET
            target.exchange_rate = source.exchange_rate,
            target._ingested_at = source._ingested_at
        WHEN NOT MATCHED THEN INSERT *
    """
    spark.sql(merge_sql)

    logger.info(f"Success! Table {target_table} updated through {end_date}.")
    record_run_result(spark, RUN_HISTORY_TABLE, TASK_NAME, status="success")

except Exception as e:
    duration = time.time() - start
    logger.error(f"BCB exchange rate ingestion failed | duration={duration:.2f}s | error={e}")
    record_run_result(spark, RUN_HISTORY_TABLE, TASK_NAME, status="failed")
    raise